Посмотрел одним глазом на соревнование https://ods.ai/competitions/ru-bashkir-lores-mt по переводу на редкие языки. 

Конкретно для башкирского языка есть 

1) хороший индикатор схожести https://huggingface.co/slone/bert-base-multilingual-cased-bak-rus-similarity 

2) предобученная модель NLLB-200

3) и корпус текстов https://huggingface.co/datasets/AigizK/bashkir-russian-parallel-corpora

К сожалению, модели "zhursvlevy/t5-small-bashkir-russian" и QWEN с малым (до 4B) числом параметров показали низкое качество перевода и плохо дообучались.

Prompt и prefix tuning это конечно прикольно, но наилучшие результаты, сравнимые с полным дообучением, показывает LoRA - прибавление к некоторым матричным весам модели дообучаемого произведения пары матриц низкого ранга, дающих матрицу такого же размера в сумме https://www.runpod.io/articles/guides/llm-fine-tuning-on-a-budget-top-faqs-on-adapters-lora-and-other-parameter-efficient-methods

Здесь я собрал небольшой пайплайн по дообучению модели-переводчика от F***book (проект M*ta Platforms, Inc., деятельность которой в России запрещена, компания M*ta Platforms Inc. признана экстремистской организацией и её деятельность запрещена в России по решению Тверского суда Москвы от 21 марта 2022 года). К сожалению, данная модель не предназначена для коммерческого использования, поэтому приз за участие в соревновании получить, строго говоря, за нее нельзя, да и ресурсы халявные на Collab и Kaggle кончились, надо в следующий раз арендовать https://datasphere.yandex.cloud/configurations , поэтому ограничился здесь небольшим улучшением качества. 



# Загружаем модель для оценки качества перевода

In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

clf_name = 'slone/bert-base-multilingual-cased-bak-rus-similarity'
clf = AutoModelForSequenceClassification.from_pretrained(clf_name)
clf_tokenizer = AutoTokenizer.from_pretrained(clf_name)

def classify(texts_ba, texts_ru):
    with torch.inference_mode():
        batch = clf_tokenizer(texts_ba, texts_ru, padding=True, truncation=True, max_length=512, return_tensors='pt').to(clf.device)
        return torch.softmax(clf(**batch).logits.view(-1, 2), -1)[:, 1].cpu().numpy()

# print(classify(['Сәләм, ғаләм!', 'Хәйерле көн, тыныслыҡ.'], ['Привет, мир!', 'Мама мыла раму.']))
print(classify(['Һаумыһығыҙ, тыныслыҡ!', 'Әсәйем рама йыуа.'], ['Привет, мир!', 'Мама мыла раму.']))
# [0.96345973 0.02213471]

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/711M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

[0.9484951  0.95168203]


# Загружаем базовую модель

In [4]:
pip install transformers torch sentencepiece -q

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

checkpoint = 'f***book/nllb-200-distilled-600M' # название зацензурено
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)

source_text = 'Привет, мир!' # Текст для проверки работоспособности модели

source_lang = 'rus_Cyrl'  # Русский
target_lang = 'bak_Cyrl'  # Башкирский

translator = pipeline(
    'translation',
    model=model,
    tokenizer=tokenizer,
    src_lang=source_lang,
    tgt_lang=target_lang,
    max_length=400
)

# Выполнение перевода
output = translator(source_text)
translated_text = output[0]['translation_text']
print(translated_text)  

2026-01-10 11:43:44.627788: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768045425.037348      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768045425.172875      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768045426.101088      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768045426.101130      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768045426.101133      55 computation_placer.cc:177] computation placer alr

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Device set to use cuda:0


Һаумыһығыҙ, тыныслыҡ!


In [4]:
print(classify(['Һаумыһығыҙ, тыныслыҡ!'], ['Привет, мир!']))

[0.9484951]


In [3]:
# Загружаем данные, которые организаторы соревнования предоставили для валидации
import pandas as pd
tst = pd.read_csv('https://storage.yandexcloud.net/ds-ods/files/content/2025/11/27/80edce61/test.csv')
sam = pd.read_csv('https://storage.yandexcloud.net/ds-ods/files/content/2025/11/27/ea0f945e/sample.csv')
tst_0 = tst[:100]

In [12]:
# Смотрим качество исходной модели на 100 примерах
output = translator(list(tst_0['source_ru']))
translated_text = [output[k]['translation_text'] for k in range(len(output))]
base_0 = classify(translated_text, list(tst_0['source_ru']))
print(base_0, '\n', sum(base_0)/len(base_0))

[0.94216496 0.9389645  0.95508325 0.9565522  0.947725   0.93056554
 0.9618979  0.9473315  0.82637745 0.9475345  0.93482393 0.9405584
 0.9565337  0.9519196  0.9521141  0.8876715  0.9533831  0.94188374
 0.88757735 0.9502151  0.95284116 0.9530169  0.95881134 0.9505701
 0.9559581  0.92925894 0.9555626  0.9570375  0.96194845 0.60330784
 0.95783514 0.9525716  0.96367824 0.9497785  0.04432354 0.8711599
 0.95441604 0.93640417 0.01888425 0.9307808  0.94525737 0.9020703
 0.94234973 0.55790246 0.9421767  0.9431328  0.9474254  0.95224273
 0.94460934 0.9559358  0.9443712  0.9444139  0.9523898  0.9206466
 0.9442598  0.8331307  0.9257817  0.9501359  0.9531899  0.93010944
 0.9465426  0.7554689  0.937873   0.93004984 0.9282589  0.95055956
 0.6337693  0.9436136  0.8887605  0.2099589  0.9386452  0.9424213
 0.95416147 0.5475815  0.9621008  0.9419501  0.9470967  0.91511714
 0.94928056 0.6033294  0.93680626 0.79273444 0.9451365  0.94211847
 0.95051646 0.938195   0.9573726  0.9418037  0.9513395  0.21133265
 

# Обучаем базовую модель

In [13]:
# pip install transformers torch datasets sentencepiece peft accelerate evaluate sacrebleu -q

In [14]:
# pip install pyarrow pandas numpy --upgrade

In [15]:
# Загружаем обучающий датасет
import pandas as pd

# Login using e.g. `huggingface-cli login` to access this dataset
df = pd.read_parquet("hf://datasets/AigizK/bashkir-russian-parallel-corpora/data/train-00000-of-00001.parquet")

In [16]:
df['corpus'].unique()

array(['https://t.me/bashkort_translate_bot', '1000 sentences',
       'bashkir encyclopedia', 'https://www.bashinform.ru/', 'bashinform',
       'little prince', 'tmx corpus'], dtype=object)

In [17]:
# После разглядывания данных было принято решение взять от всех, кроме словаря по чуть-чуть 
df_0 = pd.concat([df[df.corpus=='https://t.me/bashkort_translate_bot'][:5000], 
                 df[df.corpus=='1000 sentences'][:5000], 
                 df[df.corpus=='bashkir encyclopedia'][:5000], 
                 df[df.corpus=='https://www.bashinform.ru/'][:5000], 
                 df[df.corpus=='bashinform'][:5000], 
                 df[df.corpus=='little prince'][:5000], 
                 # df[df.corpus=='tmx corpus'][:10000], 
                 ])

In [20]:
# Приводим данные к нужному для дообучения формату

train_data = list(df_0.apply(lambda x: {"translation": {"rus_Cyrl": x['ru'], "bak_Cyrl": x['ba']}},axis=1))

from datasets import Dataset

dataset = Dataset.from_list(train_data)
split_dataset = dataset.train_test_split(test_size=0.1)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

max_length = 128
source_lang = "rus_Cyrl" 
target_lang = "bak_Cyrl"

In [24]:
# А вот и LoRA!

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
from peft import get_peft_model, LoraConfig, TaskType
import torch
import numpy as np
from datasets import Dataset

model_checkpoint = 'f***book/nllb-200-distilled-600M'# название зацензурено
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

lora_config = LoraConfig(
    r=8,                      # Ранг адаптера (Если Q - m*n матрица, то прибавим к ней 
                            # матрицу  A размера m*r умноженную на B размера r*n 
    lora_alpha=32,             # Аналог скорости обучения
    lora_dropout=0.1,         
    bias="none",              
    task_type=TaskType.SEQ_2_SEQ_LM,
    target_modules=["q_proj", "v_proj"]  # Модули для адаптации
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters() 

trainable params: 1,179,648 || all params: 616,253,440 || trainable%: 0.1914


In [25]:
# Окончательная обработка обучающих данных, включая токенизацию

def preprocess_function(examples):
    inputs = [ex[source_lang] for ex in examples["translation"]]
    targets = [ex[target_lang] for ex in examples["translation"]]

    model_inputs = tokenizer(
        inputs,
        max_length=max_length,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
            targets,
            max_length=max_length,
            truncation=True,
            padding="max_length"
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_eval = eval_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/19584 [00:00<?, ? examples/s]

Map:   0%|          | 0/2177 [00:00<?, ? examples/s]

In [27]:
# Аргументы обучения
batch_size = 8
training_args = Seq2SeqTrainingArguments(
    output_dir="./nllb-lora-finetuned",  # Директория для сохранения
    eval_strategy="steps",
    eval_steps=500,
    save_steps=1000,
    learning_rate=1e-3, # Практика показала, что для LoRA выгоднее ставить большой lr
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),  
    report_to="none",  # Измените на "wandb" или "tensorboard" при необходимости
    lr_scheduler_type="linear",
    warmup_steps=500,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',  # Использовать BLEU для выбора лучшей модели
    greater_is_better=True
)

# Коллатор данных для Seq2Seq
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

In [28]:
# Инициализация тренера
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    tokenizer=tokenizer,
    # compute_metrics=compute_metrics
)

# Старт обучения
trainer.train()

# Сохранение модели
trainer.save_model("./nllb-lora-finetuned-final")
tokenizer.save_pretrained("./nllb-lora-finetuned-final")

/tmp/ipykernel_773/4000549800.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
No label_names provided for model class `PeftModelForSeq2SeqLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.58.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss,Validation Loss
500,7.124300,5.932815
1000,5.997500,5.918404
1500,5.969800,5.912174
2000,5.975600,5.909113
2500,5.953200,5.906891
3000,5.931800,5.905420
3500,5.949400,5.903529


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


('./nllb-lora-finetuned-final/tokenizer_config.json',
 './nllb-lora-finetuned-final/special_tokens_map.json',
 './nllb-lora-finetuned-final/sentencepiece.bpe.model',
 './nllb-lora-finetuned-final/added_tokens.json',
 './nllb-lora-finetuned-final/tokenizer.json')

# Загружаем и тестируем дообученную модель

In [ ]:

from peft import PeftModel

# Загрузка базовой модели
# base_model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)

# Загрузка адаптеров LoRA
# model = PeftModel.from_pretrained(base_model, "./nllb-lora-finetuned-final")

# Или загрузка напрямую
model = AutoModelForSeq2SeqLM.from_pretrained("./nllb-lora-finetuned-final")

# Пример перевода (используйте правильные коды языков из FLORES-200)
input_text = "Мама мыла раму"
inputs = tokenizer(input_text, return_tensors="pt")

target_lang_code = "bak_Cyrl"

translator = pipeline(
    'translation',
    model=model,
    tokenizer=tokenizer,
    src_lang=source_lang,
    tgt_lang=target_lang,
    max_length=400
)

# Выполнение перевода
output = translator(source_text)
translated_text = output[0]['translation_text']
print(translated_text)

In [32]:
output = translator(list(tst_0['source_ru']))
translated_text = [output[k]['translation_text'] for k in range(len(output))]
base_0 = classify(translated_text, list(tst_0['source_ru']))
print(base_0, '\n', sum(base_0)/len(base_0))

[0.9106091  0.9389645  0.95408976 0.9578328  0.93136233 0.9425485
 0.9618979  0.9494886  0.88129044 0.9475345  0.93679076 0.9405584
 0.9565337  0.951407   0.95312107 0.8590357  0.9533831  0.9407427
 0.15345629 0.95819354 0.9492593  0.9512744  0.95881134 0.9534627
 0.9556458  0.94931924 0.9525035  0.9570375  0.96119285 0.713892
 0.94566536 0.9525716  0.96367824 0.93667465 0.9387929  0.37498465
 0.9540051  0.93827635 0.01888425 0.65560895 0.95926815 0.8675968
 0.9418068  0.8850412  0.9421767  0.94746834 0.9441693  0.95271057
 0.9427985  0.95712376 0.95817    0.9446477  0.95338494 0.9206466
 0.9389155  0.8995862  0.9169052  0.95273125 0.9531899  0.9434101
 0.94952714 0.77853835 0.93555814 0.9487738  0.9282589  0.9520835
 0.89123607 0.9436136  0.94122314 0.9313927  0.9035968  0.9424213
 0.95416147 0.6508251  0.9621008  0.9480189  0.94599575 0.9322078
 0.9440642  0.90277094 0.9375403  0.9384002  0.93698245 0.9401947
 0.95051646 0.7291042  0.9573726  0.9340497  0.9502011  0.93056065
 0.94376

Ура-ура, мы видим небольшой прирост качества)